# Analytics Slash Assistant (Colab MVP)

This project is a notebook-based analytics assistant that allows users to run data analysis workflows through slash commands instead of writing Python code manually.

## Supported capabilities
- Dataset profiling
- Data quality checks
- Conservative cleaning
- Exploratory data analysis
- KPI summaries
- Root-cause investigation prompts
- Basic visualizations
- Executive summaries
- Final markdown reports

## Example commands
- `/help`
- `/goal understand why sales dropped`
- `/profile`
- `/data-quality`
- `/eda`
- `/summary`
- `/report`

## Current scope
- Google Colab notebook
- CSV input only
- Python-based workflows
- No Claude API integration yet

In [14]:
# =========================
# SETUP
# =========================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional

%matplotlib inline
sns.set(style="whitegrid")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

print("Setup complete.")

Setup complete.


In [15]:
# =========================
# SESSION STATE
# =========================

@dataclass
class ProjectSession:
    raw_df: Optional[pd.DataFrame] = None
    clean_df: Optional[pd.DataFrame] = None
    active_df_name: str = "raw_df"

    business_goal: str = ""
    preferred_output: str = ""

    schema_summary: Dict[str, Any] = field(default_factory=dict)
    data_quality_report: Dict[str, Any] = field(default_factory=dict)
    cleaning_log: List[str] = field(default_factory=list)
    eda_report: Dict[str, Any] = field(default_factory=dict)
    kpi_report: Dict[str, Any] = field(default_factory=dict)
    root_cause_report: Dict[str, Any] = field(default_factory=dict)
    summary_report: str = ""
    final_report: str = ""

    command_history: List[str] = field(default_factory=list)

session = ProjectSession()

print("Session initialized.")

Session initialized.


In [16]:
# =========================
# UTILITY FUNCTIONS
# =========================

def print_header(title: str):
    print("\n" + "=" * 80)
    print(title.upper())
    print("=" * 80)

def print_subheader(title: str):
    print("\n" + "-" * 60)
    print(title)
    print("-" * 60)

def require_data():
    if session.raw_df is None:
        raise ValueError("No dataset loaded yet. Please upload a CSV first.")

def get_active_df():
    if session.clean_df is not None:
        return session.clean_df
    return session.raw_df

def safe_preview(df, n=5, title="Preview"):
    print_subheader(title)
    display(df.head(n))

def try_parse_dates(df):
    df_copy = df.copy()
    for col in df_copy.columns:
        if df_copy[col].dtype == "object":
            try:
                parsed = pd.to_datetime(df_copy[col], errors="raise")
                if parsed.notna().sum() > 0:
                    df_copy[col] = parsed
            except:
                pass
    return df_copy

In [17]:
# =========================
# DATA UPLOAD AND LOADING
# =========================

from google.colab import files

def upload_csv():
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)

    session.raw_df = df
    session.clean_df = None
    session.active_df_name = "raw_df"
    session.command_history.append(f"/load {filename}")

    print(f"Loaded file: {filename}")
    print(f"Shape: {df.shape}")
    safe_preview(df, title="Loaded Data Preview")

In [18]:
# =========================
# COMMAND PARSER
# =========================

def parse_command(command_text: str):
    command_text = command_text.strip()

    if not command_text.startswith("/"):
        raise ValueError("Command must start with '/'. Example: /profile")

    parts = command_text.split(" ", 1)
    command = parts[0].lower()
    args = parts[1].strip() if len(parts) > 1 else ""

    return command, args

In [19]:
# =========================
# WORKFLOW: HELP
# =========================

def run_help():
    print_header("Available Commands")
    print("""
/help
    Show all supported commands

/goal <business question>
    Set the business objective

/profile
    Inspect schema and dataset structure

/data-quality
    Run missing value, duplicate, and outlier checks

/clean
    Apply conservative cleaning steps

/eda
    Run exploratory analysis

/kpi
    Generate KPI summaries for numeric columns

/root-cause <issue>
    Generate possible diagnostic leads

/visualize <optional request>
    Create basic charts

/summary
    Generate a stakeholder-friendly summary

/report
    Generate the final markdown report
""")

In [20]:
# =========================
# WORKFLOW: GOAL
# =========================

def run_goal(goal_text: str):
    if not goal_text:
        raise ValueError("Please provide a business goal after /goal")

    session.business_goal = goal_text
    session.command_history.append(f"/goal {goal_text}")

    print_header("Business Goal Saved")
    print(session.business_goal)

    system_prompt = """
    You are an analytics planning assistant.
    Rewrite the user's business goal into:
    1. Business objective
    2. Likely metrics of interest
    3. Likely dimensions to analyze
    4. Recommended next commands
    Keep it concise, structured, and business-friendly.
    Do not invent dataset-specific facts.
    """

    user_prompt = f"User business goal: {goal_text}"

    brief = call_claude(system_prompt, user_prompt, max_tokens=500)

    print_subheader("Claude Analysis Brief")
    print(brief)

In [21]:
# =========================
# WORKFLOW: PROFILE
# =========================

def run_profile():
    require_data()
    df = get_active_df()

    profile_df = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[col].dtype) for col in df.columns],
        "missing_count": [df[col].isnull().sum() for col in df.columns],
        "missing_pct": [round(df[col].isnull().mean() * 100, 2) for col in df.columns],
        "unique_values": [df[col].nunique(dropna=False) for col in df.columns]
    })

    session.schema_summary = {
        "shape": df.shape,
        "profile_table": profile_df
    }
    session.command_history.append("/profile")

    print_header("Dataset Profile")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    display(profile_df)
    print("\nNext suggested steps:")
    print("- Run /data-quality to check data issues")
    print("- Run /eda to explore patterns")

In [22]:
# =========================
# WORKFLOW: DATA QUALITY
# =========================

def run_data_quality():
    require_data()
    df = get_active_df()

    duplicate_count = int(df.duplicated().sum())

    missing_df = pd.DataFrame({
        "column": df.columns,
        "missing_count": [df[col].isnull().sum() for col in df.columns],
        "missing_pct": [round(df[col].isnull().mean() * 100, 2) for col in df.columns]
    }).sort_values(by="missing_pct", ascending=False)

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    outlier_summary = {}

    for col in numeric_cols:
        series = df[col].dropna()
        if len(series) > 0:
            q1 = series.quantile(0.25)
            q3 = series.quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            outliers = ((series < lower) | (series > upper)).sum()
            outlier_summary[col] = int(outliers)

    session.data_quality_report = {
        "duplicate_count": duplicate_count,
        "missing_table": missing_df,
        "outlier_summary": outlier_summary
    }
    session.command_history.append("/data-quality")

    print_header("Data Quality Report")
    print(f"Duplicate rows: {duplicate_count}")

    print_subheader("Missing Value Summary")
    display(missing_df)

    print_subheader("Potential Outliers")
    if outlier_summary:
        display(pd.DataFrame({
            "column": list(outlier_summary.keys()),
            "outlier_count": list(outlier_summary.values())
        }).sort_values(by="outlier_count", ascending=False))
    else:
        print("No numeric columns found.")

    print("\nNext suggested steps:")
    print("- Run /clean to fix issues")
    print("- Run /eda to analyze cleaned data")

In [23]:
# =========================
# WORKFLOW: CLEAN
# =========================

def run_clean():
    require_data()
    df = session.raw_df.copy()
    cleaning_log = []

    original_shape = df.shape

    df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
    cleaning_log.append("Standardized column names.")

    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    cleaning_log.append(f"Removed {before - after} duplicate rows.")

    object_cols = df.select_dtypes(include=["object"]).columns
    for col in object_cols:
        df[col] = df[col].astype(str).str.strip()
    cleaning_log.append("Trimmed whitespace in text columns.")

    df = try_parse_dates(df)
    cleaning_log.append("Attempted date parsing on object columns.")

    session.clean_df = df
    session.active_df_name = "clean_df"
    session.cleaning_log = cleaning_log
    session.command_history.append("/clean")

    print_header("Cleaning Complete")
    print(f"Original shape: {original_shape}")
    print(f"Cleaned shape: {df.shape}")

    print_subheader("Cleaning Log")
    for item in cleaning_log:
        print(f"- {item}")

    safe_preview(df, title="Cleaned Data Preview")

In [24]:
# =========================
# WORKFLOW: EDA
# =========================

def run_eda():
    require_data()
    df = get_active_df()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    datetime_cols = df.select_dtypes(include=["datetime64[ns]"]).columns.tolist()

    report = {}

    if numeric_cols:
        report["numeric_summary"] = df[numeric_cols].describe().T
    else:
        report["numeric_summary"] = None

    top_categories = {}
    for col in categorical_cols[:5]:
        top_categories[col] = df[col].value_counts(dropna=False).head(10)

    report["top_categories"] = top_categories
    report["datetime_columns"] = datetime_cols

    session.eda_report = report
    session.command_history.append("/eda")

    print_header("EDA Report")

    if report["numeric_summary"] is not None:
        print_subheader("Numeric Summary")
        display(report["numeric_summary"])
    else:
        print("No numeric columns found.")

    if top_categories:
        print_subheader("Top Categories")
        for col, series in top_categories.items():
            print(f"\nColumn: {col}")
            display(series.to_frame(name="count"))

    if datetime_cols:
        print_subheader("Detected Date Columns")
        print(datetime_cols)

    print("\nNext suggested steps:")
    print("- Run /kpi to summarize metrics")
    print("- Run /root-cause <issue> for deeper analysis")

In [25]:
# =========================
# WORKFLOW: KPI
# =========================

def run_kpi():
    require_data()
    df = get_active_df()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric_cols:
        print("No numeric columns available for KPI generation.")
        return

    rows = []
    for col in numeric_cols:
        rows.append({
            "metric": col,
            "count": df[col].count(),
            "sum": df[col].sum(),
            "mean": df[col].mean(),
            "median": df[col].median(),
            "min": df[col].min(),
            "max": df[col].max()
        })

    kpi_df = pd.DataFrame(rows)
    session.kpi_report = {"kpi_table": kpi_df}
    session.command_history.append("/kpi")

    print_header("KPI Report")
    display(kpi_df)

In [26]:
# =========================
# WORKFLOW: ROOT CAUSE
# =========================

def run_root_cause(issue_text: str):
    require_data()
    df = get_active_df()

    if not issue_text:
        raise ValueError("Please provide an issue. Example: /root-cause sales decline")

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

    insights = []

    if numeric_cols:
        variances = df[numeric_cols].var().sort_values(ascending=False)
        insights.append("High-variance numeric columns worth investigating:")
        insights.extend([f"- {col}" for col in variances.head(5).index.tolist()])

    if categorical_cols:
        insights.append("Categorical dimensions worth segmenting by:")
        insights.extend([f"- {col}" for col in categorical_cols[:5]])

    session.root_cause_report = {
        "issue": issue_text,
        "possible_drivers": insights
    }
    session.command_history.append(f"/root-cause {issue_text}")

    print_header("Root Cause Investigation")
    print(f"Issue: {issue_text}")
    print()
    for item in insights:
        print(item)

    print("\nNote: These are diagnostic leads, not causal conclusions.")

In [27]:
# =========================
# WORKFLOW: VISUALIZE
# =========================

def run_visualize(request_text: str = ""):
    require_data()
    df = get_active_df()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

    print_header("Visualization")

    if request_text:
        print(f"User request: {request_text}")

    if numeric_cols:
        col = numeric_cols[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[col].dropna(), kde=True)
        plt.title(f"Distribution of {col}")
        plt.show()

    if categorical_cols:
        col = categorical_cols[0]
        plt.figure(figsize=(8, 4))
        df[col].value_counts().head(10).plot(kind="bar")
        plt.title(f"Top 10 Categories in {col}")
        plt.show()

    session.command_history.append(f"/visualize {request_text}".strip())

In [28]:
# =========================
# WORKFLOW: SUMMARY
# =========================

def run_summary():
    require_data()

    shape = session.schema_summary.get("shape", ("unknown", "unknown"))
    duplicate_count = session.data_quality_report.get("duplicate_count", 0)
    kpi_table = session.kpi_report.get("kpi_table")
    root_issue = session.root_cause_report.get("issue", "")

    structured_context = f"""
Business goal: {session.business_goal}
Dataset shape: {shape}
Duplicate rows: {duplicate_count}
Root cause issue investigated: {root_issue}

KPI table:
{kpi_table.to_string(index=False) if kpi_table is not None else 'No KPI table generated'}

Cleaning log:
{chr(10).join(session.cleaning_log) if session.cleaning_log else 'No cleaning steps recorded'}
"""

    system_prompt = """
    You are a business analytics assistant.
    Write a concise executive summary for a non-technical stakeholder.
    Use only the facts provided.
    Do not invent numbers.
    Do not make causal claims unless clearly supported.
    Keep the tone crisp and business-friendly.
    """

    summary_text = call_claude(system_prompt, structured_context, max_tokens=700)

    session.summary_report = summary_text
    session.command_history.append("/summary")

    print_header("Executive Summary")
    print(summary_text)

In [29]:

# =========================
# WORKFLOW: REPORT
# =========================

def run_report():
    require_data()

    shape = session.schema_summary.get("shape", ("unknown", "unknown"))
    duplicate_count = session.data_quality_report.get("duplicate_count", 0)
    kpi_table = session.kpi_report.get("kpi_table")
    root_issue = session.root_cause_report.get("issue", "")

    structured_context = f"""
Business goal: {session.business_goal}
Dataset shape: {shape}
Duplicate rows: {duplicate_count}
Cleaning log: {session.cleaning_log}
Root cause issue investigated: {root_issue}

KPI table:
{kpi_table.to_string(index=False) if kpi_table is not None else 'No KPI table generated'}

Existing executive summary:
{session.summary_report if session.summary_report else 'No summary generated yet'}
"""

    system_prompt = """
    You are a business analytics reporting assistant.
    Write a final markdown report with these sections:
    1. Business Goal
    2. Dataset Overview
    3. Data Quality Notes
    4. Key Findings
    5. Caveats
    6. Recommended Next Steps

    Rules:
    - Use only provided facts
    - Do not invent numbers
    - Do not claim causality unless explicitly supported
    - Write clearly for business stakeholders
    """

    report_text = call_claude(system_prompt, structured_context, max_tokens=1200)

    session.final_report = report_text
    session.command_history.append("/report")

    print_header("Final Report")
    print(report_text)

## Claude Integration
This section adds Claude as an intelligence layer for business-goal interpretation, summaries, and report generation.

In [30]:
!pip -q install anthropic

In [31]:
import os
import anthropic

In [32]:
# =========================
# CLAUDE CONFIG
# =========================

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
USE_CLAUDE = bool(ANTHROPIC_API_KEY)

if USE_CLAUDE:
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    print("Claude client initialized.")
else:
    client = None
    print("No API key found. Running in no-Claude mode.")

No API key found. Running in no-Claude mode.


In [33]:
# =========================
# CLAUDE HELPER
# =========================

def call_claude(system_prompt: str, user_prompt: str, max_tokens: int = 800):
    if not USE_CLAUDE:
        return "[Claude not connected yet. Placeholder response.]"

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.content[0].text

In [8]:
# =========================
# COMMAND RUNNER
# =========================

def run_command(command_text: str):
    command, args = parse_command(command_text)

    if command == "/help":
        run_help()
    elif command == "/goal":
        run_goal(args)
    elif command == "/profile":
        run_profile()
    elif command == "/data-quality":
        run_data_quality()
    elif command == "/clean":
        run_clean()
    elif command == "/eda":
        run_eda()
    elif command == "/kpi":
        run_kpi()
    elif command == "/root-cause":
        run_root_cause(args)
    elif command == "/visualize":
        run_visualize(args)
    elif command == "/summary":
        run_summary()
    elif command == "/report":
        run_report()
    else:
        print(f"Unknown command: {command}")
        print("Run /help to see supported commands.")

In [ ]:
# =========================
# EXAMPLE USAGE
# =========================

# Step 1: Upload CSV
# upload_csv()

# Step 2: Run commands
# run_command("/help")
# run_command("/goal understand why sales dropped")
# run_command("/profile")
# run_command("/data-quality")
# run_command("/clean")
# run_command("/eda")
# run_command("/kpi")
# run_command("/root-cause sales decline")
# run_command("/visualize")
# run_command("/summary")
# run_command("/report")

In [34]:
upload_csv()

Saving sample_sales_data.csv to sample_sales_data (2).csv
Loaded file: sample_sales_data (2).csv
Shape: (15, 7)

------------------------------------------------------------
Loaded Data Preview
------------------------------------------------------------


,date,region,product,category,sales,quantity,price
0,2024-01-01,North,A,Electronics,1200,2,600
1,2024-01-02,South,B,Furniture,800,1,800
2,2024-01-03,East,A,Electronics,1500,3,500
3,2024-01-04,West,C,Clothing,400,4,100
4,2024-01-05,North,B,Furniture,1000,2,500


## Run the assistant

1. Run the upload cell first  
2. Then enter one slash command at a time below

In [ ]:
# =========================
# USER INPUT INTERFACE
# =========================

user_input = input("Enter command (type /help to see options): ")

run_command(user_input)

Enter command (type /help to see options): /profile

DATASET PROFILE
Rows: 15
Columns: 7


,column,dtype,missing_count,missing_pct,unique_values
0,date,object,0,0.0,15
1,region,object,0,0.0,4
2,product,object,0,0.0,3
3,category,object,0,0.0,3
4,sales,int64,0,0.0,15
5,quantity,int64,0,0.0,5
6,price,int64,0,0.0,10



Next suggested steps:
- Run /data-quality to check data issues
- Run /eda to explore patterns


In [ ]:
def run_status():
    print_header("Session Status")
    print(f"Business goal set: {'Yes' if session.business_goal else 'No'}")
    print(f"Raw dataset loaded: {'Yes' if session.raw_df is not None else 'No'}")
    print(f"Cleaned dataset available: {'Yes' if session.clean_df is not None else 'No'}")
    print(f"Profile completed: {'Yes' if bool(session.schema_summary) else 'No'}")
    print(f"Data quality completed: {'Yes' if bool(session.data_quality_report) else 'No'}")
    print(f"EDA completed: {'Yes' if bool(session.eda_report) else 'No'}")
    print(f"KPI analysis completed: {'Yes' if bool(session.kpi_report) else 'No'}")
    print(f"Root cause analysis completed: {'Yes' if bool(session.root_cause_report) else 'No'}")
    print(f"Summary generated: {'Yes' if bool(session.summary_report) else 'No'}")
    print(f"Report generated: {'Yes' if bool(session.final_report) else 'No'}")

In [ ]:
def run_command(command_text: str):
    command, args = parse_command(command_text)

    if command == "/help":
        run_help()
    elif command == "/goal":
        run_goal(args)
    elif command == "/status":
        run_status()
    elif command == "/profile":
        run_profile()
    elif command == "/data-quality":
        run_data_quality()
    elif command == "/clean":
        run_clean()
    elif command == "/eda":
        run_eda()
    elif command == "/kpi":
        run_kpi()
    elif command == "/root-cause":
        run_root_cause(args)
    elif command == "/visualize":
        run_visualize(args)
    elif command == "/summary":
        run_summary()
    elif command == "/report":
        run_report()
    else:
        print(f"Unknown command: {command}")
        print("Run /help to see supported commands.")

In [ ]:
upload_csv()

Saving sample_sales_data.csv to sample_sales_data (2).csv
Loaded file: sample_sales_data (2).csv
Shape: (15, 7)

------------------------------------------------------------
Loaded Data Preview
------------------------------------------------------------


,date,region,product,category,sales,quantity,price
0,2024-01-01,North,A,Electronics,1200,2,600
1,2024-01-02,South,B,Furniture,800,1,800
2,2024-01-03,East,A,Electronics,1500,3,500
3,2024-01-04,West,C,Clothing,400,4,100
4,2024-01-05,North,B,Furniture,1000,2,500
